# Recall @ Top-N% across architectures
Builds the cross-model recall table from `sensitivity_measurement` W&B runs: for every
proxy metric, the fraction of the truly most sensitive layers it recovers.

In [ ]:
ENTITY = None
PROJECT = "???"
# One run id per architecture. Order here is the column order of the final table.
RUNS = {
    "ResNet-18": "???",
    "ConvNeXt-Tiny": "???",
    "EfficientNet-B0": "???",
    "TinyViT": "???",
}
GROUND_TRUTH_COLUMN = "acc_drop"
FRACTIONS = (0.10, 0.20, 0.30)

In [ ]:
import json
import logging
import os
import tempfile

import pandas as pd
import rootutils

root = rootutils.setup_root(".", indicator=".project-root", pythonpath=True, cwd=True)

os.environ.setdefault("WANDB_SILENT", "true")
import wandb
from omegaconf import OmegaConf

from src.flows.sensitivity_measurement.recall_top_k import _build_layer_table
from src.metrics.ranking import compute_recall_at_k

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
api = wandb.Api()
entity = ENTITY or api.default_entity

## Predictors

In [ ]:
recall_cfg = OmegaConf.load(root / "config" / "sensitivity_measurement.yaml").recall_top_k
PREDICTORS = {str(k): bool(v) for k, v in OmegaConf.to_container(recall_cfg.predictors).items()}

pd.DataFrame(
    [{"predictor": k, "higher = more sensitive": v} for k, v in PREDICTORS.items()]
).set_index("predictor")

## Fetch each run's tables

The three steps log their tables as W&B media files. They are written out under the
filenames `_build_layer_table` expects, so the merge — including the ONNX-to-PyTorch
name bridge for HAWQ-V2 — is the same code the flow runs.

In [ ]:
TABLE_FILES = {
    "oat/results": "oat_sensitivity_results.csv",
    "weights/per_node_stats": "weights_per_node_stats.csv",
    "hawqv2_scores": "hawqv2_scores.csv",
}


def _read_wandb_table(path: str) -> pd.DataFrame:
    """W&B writes tables either as orient='split' or as a {columns, data} payload."""
    try:
        return pd.read_json(path, orient="split")
    except ValueError:
        payload = json.load(open(path))
        return pd.DataFrame(data=payload["data"], columns=payload["columns"])


def fetch_run_tables(run_path: str, dest: str) -> dict:
    """Download a run's step tables into `dest` under the flow's CSV names.

    Returns the run's logged config, which says how the run was measured.
    """
    run = api.run(run_path)
    written = []
    for f in run.files():
        if not f.name.endswith(".table.json"):
            continue
        stem = f.name.replace("media/table/", "").rsplit("_", 2)[0]
        if stem not in TABLE_FILES:
            continue
        local = f.download(root=dest, replace=True)
        _read_wandb_table(local.name).to_csv(os.path.join(dest, TABLE_FILES[stem]), index=False)
        written.append(TABLE_FILES[stem])
    return dict(run.config)


def local_profiling_filter(run_config: dict):
    """The run's profiling filter, resolved against this checkout.

    The config stores the absolute path on the machine that ran it; the node
    lists ship with the repo under data/profiling/, so only the file name is
    kept. None when the run was not filtered.
    """
    configured = (run_config.get("recall_top_k") or {}).get("profiling_filter")
    if not configured:
        return None
    local = root / "data" / "profiling" / os.path.basename(str(configured))
    if not local.exists():
        raise FileNotFoundError(f"Profiling node list not found: {local}")
    return str(local)

In [ ]:
layer_tables, recalls, provenance = {}, {}, {}
for label, run_id in RUNS.items():
    run_path = f"{entity}/{PROJECT}/{run_id}"
    print(f"\n=== {label} ===")
    with tempfile.TemporaryDirectory() as tmp:
        run_config = fetch_run_tables(run_path, tmp)
        profiling_filter = local_profiling_filter(run_config)
        df = _build_layer_table(tmp, profiling_filter)
    max_test_batches = run_config.get("max_test_batches", "not logged")
    provenance[label] = {
        "run": run_path.rsplit("/", 1)[-1],
        "max_test_batches": max_test_batches,
        "profiling_filter": os.path.basename(profiling_filter) if profiling_filter else None,
    }
    if max_test_batches is not None:
        print(
            f"  WARNING: max_test_batches={max_test_batches}. The ground truth is not "
            f"measured on the whole test set, so this column is not comparable with "
            f"the paper (on ResNet-18 and TinyViT it is mostly noise)."
        )
    if df is None:
        print(f"  skipped: the run has no OAT table")
        continue
    layer_tables[label] = df
    recalls[label] = compute_recall_at_k(df, PREDICTORS, GROUND_TRUTH_COLUMN, FRACTIONS)
    provenance[label]["layers"] = len(df)
    print(f"  {len(df)} layers, k = {[max(1, int(len(df) * f)) for f in FRACTIONS]}")

pd.DataFrame(provenance).T

## Recall table

In [ ]:
COLS = [f"recall@{int(f * 100)}%" for f in FRACTIONS]

table = pd.concat(
    {label: r[COLS] for label, r in recalls.items()}, axis=1
).reindex(PREDICTORS.keys()).dropna(how="all")
table.columns = pd.MultiIndex.from_tuples(
    [(m, c.replace("recall", "")) for m, c in table.columns]
)
table[("", "Avg.")] = table.mean(axis=1).round(3)
table = table.sort_values(("", "Avg."), ascending=False)

k_row = {label: [max(1, int(len(df) * f)) for f in FRACTIONS] for label, df in layer_tables.items()}
print("k per column:", k_row)
table.style.format("{:.3f}").background_gradient(cmap="Blues", axis=None)